In [3]:
# Import Libraries
import pandas as pd
import numpy as np
import nltk
import string

from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Download NLTK data
nltk.download('stopwords')

import kagglehub
import os

# Download latest version
path = kagglehub.dataset_download("yufengdev/bbc-fulltext-and-category")

print("Path to dataset files:", path)

# Load Dataset from the downloaded path
df = pd.read_csv(os.path.join(path, "bbc-text.csv"))

print(df.head())
print(df['category'].value_counts())

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


Using Colab cache for faster access to the 'bbc-fulltext-and-category' dataset.
Path to dataset files: /kaggle/input/bbc-fulltext-and-category
        category                                               text
0           tech  tv future in the hands of viewers with home th...
1       business  worldcom boss  left books alone  former worldc...
2          sport  tigers wary of farrell  gamble  leicester say ...
3          sport  yeading face newcastle in fa cup premiership s...
4  entertainment  ocean s twelve raids box office ocean s twelve...
category
sport            511
business         510
politics         417
tech             401
entertainment    386
Name: count, dtype: int64


In [4]:
stemmer = PorterStemmer()
stop_words = set(stopwords.words('english'))

def preprocess(text):
    text = text.lower()

    # Remove punctuation
    text = text.translate(str.maketrans('', '', string.punctuation))

    words = text.split()

    words = [stemmer.stem(word) for word in words if word not in stop_words]

    return " ".join(words)

df["clean_text"] = df["text"].apply(preprocess)

print(df.head())

        category                                               text  \
0           tech  tv future in the hands of viewers with home th...   
1       business  worldcom boss  left books alone  former worldc...   
2          sport  tigers wary of farrell  gamble  leicester say ...   
3          sport  yeading face newcastle in fa cup premiership s...   
4  entertainment  ocean s twelve raids box office ocean s twelve...   

                                          clean_text  
0  tv futur hand viewer home theatr system plasma...  
1  worldcom boss left book alon former worldcom b...  
2  tiger wari farrel gambl leicest say rush make ...  
3  yead face newcastl fa cup premiership side new...  
4  ocean twelv raid box offic ocean twelv crime c...  


In [5]:
tfidf = TfidfVectorizer(max_features=5000)

X = tfidf.fit_transform(df["clean_text"])

y = df["category"]

In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [7]:
model = MultinomialNB()

model.fit(X_train, y_train)

MultinomialNB()

In [8]:
y_pred = model.predict(X_test)

In [9]:
print("Accuracy:", accuracy_score(y_test, y_pred))

Accuracy: 0.9617977528089887


In [10]:
print(classification_report(y_test, y_pred))

               precision    recall  f1-score   support

     business       0.96      0.94      0.95       101
entertainment       1.00      0.90      0.95        81
     politics       0.91      0.98      0.94        83
        sport       0.99      1.00      0.99        98
         tech       0.95      0.99      0.97        82

     accuracy                           0.96       445
    macro avg       0.96      0.96      0.96       445
 weighted avg       0.96      0.96      0.96       445



In [11]:
print(confusion_matrix(y_test, y_pred))

[[95  0  6  0  0]
 [ 2 73  2  0  4]
 [ 2  0 81  0  0]
 [ 0  0  0 98  0]
 [ 0  0  0  1 81]]


In [12]:
article = """
India won the cricket match by 7 wickets and qualified for the finals.
"""

clean = preprocess(article)

vector = tfidf.transform([clean])

prediction = model.predict(vector)

print("Predicted Category:", prediction[0])

Predicted Category: sport
